In [ ]:
# Step 1
# Install bleeding edge dependencies transformers and PEFT
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q git+https://github.com/huggingface/peft.git
!pip install -q accelerate bitsandbytes datasets numpy trl

In [ ]:
# Step 2
# Imports
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

MODEL_ID = "google/gemma-4-e4b"
LORA_RANK = 16
LORA_ALPHA = 32
POPULATION_SIZE = 32
LEARNING_RATE = 0.05
NOISE_STD = 0.02
GENERATIONS = 50
BATCH_SIZE = 4

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# Step 3
# Load Base Model in 4-bit precision
# Inject LoRA Adapters
from transformers import BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map={"": 0},
    quantization_config=bnb_config,
    dtype=torch.float16
)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj.linear", "v_proj.linear", "k_proj.linear", "o_proj.linear"],
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

essa_params = []

def extract_and_reparameterize(module, rank):
    device = module.lora_A.default.weight.device

    # IMPROVEMENT: Initialize B to non-zero to prevent zero singular values
    torch.nn.init.normal_(module.lora_B.default.weight, mean=0.0, std=0.02)

    A = module.lora_A.default.weight.data.float().cpu()
    B = module.lora_B.default.weight.data.float().cpu()

    W = B @ A
    # IMPROVEMENT: Use non-deprecated linalg.svd
    U, S, Vh = torch.linalg.svd(W, full_matrices=False)
    V = Vh.T

    U_r = U[:, :rank].to(device)
    S_r = S[:rank].to(device)
    V_r = V[:, :rank].to(device)

    # IMPROVEMENT: Removed unnecessary gradient tracking (S_r.requires_grad = True)
    return {"module": module, "U": U_r, "S": S_r, "V": V_r}

def apply_singular_values(param_dict, current_S):
    U, V, module = param_dict["U"], param_dict["V"], param_dict["module"]
    S_safe = torch.relu(current_S) + 1e-8
    S_sqrt = torch.diag(torch.sqrt(S_safe))

    module.lora_B.default.weight.data = (U @ S_sqrt).to(module.lora_B.default.weight.dtype)
    module.lora_A.default.weight.data = (S_sqrt @ V.T).to(module.lora_A.default.weight.dtype)

for name, module in model.named_modules():
    if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
        essa_params.append(extract_and_reparameterize(module, LORA_RANK))

In [ ]:
# Step 4
from datasets import load_dataset
from transformers import pipeline
import numpy as np

dataset = load_dataset("Maximofn/short-jokes-dataset", split="train[:1000]")

def get_batch(batch_size):
    indices = np.random.choice(len(dataset), batch_size, replace=False)
    return [dataset[int(i)] for i in indices]

# IMPROVEMENT: Swapped to a preference reward model instead of SST-2
reward_model = pipeline(
    "text-classification",
    model="OpenAssistant/reward-model-deberta-v3-base",
    device=0
)

def evaluate_humor_reward_batch(responses):
    safe_responses = [resp[:512] if resp.strip() else "empty" for resp in responses]
    results = reward_model(safe_responses, truncation=True, batch_size=len(safe_responses))
    
    scores = []
    for response, result in zip(responses, results):
        if not response.strip() or response == "empty":
            scores.append(0.0)
            continue
        
        score = result['score']
        if len(response) < 10:
            score -= 0.5
        scores.append(score)
    
    return scores

In [ ]:
# Step 5
# Batched Reward ESSA Loop
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# IMPROVEMENT: Use a fixed reference batch for stability across generations
eval_batch = get_batch(BATCH_SIZE)
prompts = [f"System: You are a comedian. Make this funnier: {sample.get('Joke', 'Tell me a joke.')}\nAnswer:" for sample in eval_batch]
inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to("cuda")
prompt_lengths = inputs.input_ids.shape[1]

for generation in range(GENERATIONS):
    population_noise = [[torch.randn_like(p["S"]) * NOISE_STD for p in essa_params] for _ in range(POPULATION_SIZE)]
    rewards = np.zeros(POPULATION_SIZE)

    # IMPROVEMENT: Calculate baseline reward for cleaner gradient signal
    for param in essa_params:
        apply_singular_values(param, param["S"])
    with torch.no_grad():
        base_outputs = model.generate(**inputs, max_new_tokens=64, do_sample=True, temperature=0.7, pad_token_id=tokenizer.pad_token_id)
    base_responses = tokenizer.batch_decode(base_outputs[:, prompt_lengths:], skip_special_tokens=True)
    baseline_reward = np.mean(evaluate_humor_reward_batch(base_responses))

    for p_idx, noise in enumerate(population_noise):
        for i, param in enumerate(essa_params):
            apply_singular_values(param, param["S"] + noise[i])

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=64, do_sample=True, temperature=0.7, pad_token_id=tokenizer.pad_token_id)

        generated_tokens = outputs[:, prompt_lengths:]
        responses = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        batch_rewards = evaluate_humor_reward_batch(responses)
        rewards[p_idx] = np.mean(batch_rewards)

    # IMPROVEMENT: Baseline subtraction instead of standardizing across the population
    normalized_rewards = rewards - baseline_reward
    print(f"Gen {generation + 1}/{GENERATIONS} | Baseline Reward: {baseline_reward:.4f} | Max Reward: {np.max(rewards):.4f}")

    with torch.no_grad():
        for i, param in enumerate(essa_params):
            grad = sum(normalized_rewards[p_idx] * population_noise[p_idx][i] for p_idx in range(POPULATION_SIZE))
            param["S"].data += LEARNING_RATE * (grad / (POPULATION_SIZE * NOISE_STD))
            apply_singular_values(param, param["S"])

In [ ]:
# Save the final LoRA adapters
# IMPROVEMENT: Corrected output directory to reflect the joke dataset
output_dir = "./gemma4-e4b-essa-jokes"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"ESSA optimized adapters saved to {output_dir}")